# Quick Start: Running Toto 2.0 on BOOM benchmark

This notebook is adapted from the [GiftEval repository](https://github.com/SalesforceAIResearch/gift-eval/tree/main/notebooks) and shows how to run Toto 2.0 on the BOOM benchmark using the built-in `Toto2GluonTSModel` wrapper, which produces a standard gluonts `PyTorchPredictor`.

`context_length` is set to 2048 for all evaluations in this notebook.

Make sure you download the BOOM benchmark and set the `BOOM` environment variable correctly before running this notebook.

We will use the `Dataset` class from GiftEval to load the data and run the model.

## Download BOOM

`download_boom_benchmark` also sets the `BOOM` environment variable with the correct path, which is needed for running the evals below.

In [ ]:
import os
import json
from dotenv import load_dotenv
from dataset_utils import download_boom_benchmark

boom_path = "ChangeMe"
download_boom_benchmark(boom_path)
load_dotenv()

dataset_properties_map = json.load(open("./boom/boom_properties.json"))
all_datasets = list(dataset_properties_map.keys())
print(len(all_datasets))

In [ ]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

## Load Toto 2.0

Available sizes: `4m | 22m | 313m | 1B | 2.5B`. The base model is loaded once and re-wrapped per dataset with a `Toto2GluonTSModel`. The wrapper's `create_predictor()` returns a standard gluonts `PyTorchPredictor` that handles transforms, instance splitting, and batched inference.

In [ ]:
import torch
from toto2 import Toto2Model, Toto2GluonTSModel, Toto2GluonTSModelConfig

SIZE = "22m"  # 4m | 22m | 313m | 1B | 2.5B
CHECKPOINT = f"Datadog/Toto-2.0-{SIZE}"
CONTEXT_LENGTH = 2048

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_float32_matmul_precision("high")

toto_model = Toto2Model.from_pretrained(CHECKPOINT, map_location=DEVICE).to(DEVICE).eval()
print(f"Loaded {CHECKPOINT}: {sum(p.numel() for p in toto_model.parameters()):,} params")

## Evaluation

Iterate over all BOOM datasets and terms, rebuild a predictor per dataset (to pick up the dataset's `prediction_length` and `target_dim`), and record metrics to `results/toto2/all_results.csv`.

Toto 2.0's `PyTorchPredictor` emits `QuantileForecast`s (no mean stored), so we filter the corresponding gluonts warning. The `MSE[mean]` / `RMSE[mean]` columns may be NaN as a result — use the 0.5-quantile variants for a point-metric comparison.

In [ ]:
import csv
import logging

from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset

class _Filter(logging.Filter):
    def filter(self, record):
        return "The mean prediction is not stored in the forecast data" not in record.getMessage()
logging.getLogger("gluonts.model.forecast").addFilter(_Filter())

model_name = "toto2"
output_dir = f"../results/{model_name}"
os.makedirs(output_dir, exist_ok=True)
csv_file_path = os.path.join(output_dir, "all_results.csv")

BATCH_SIZE = 64  # lower this if you OOM on big multivariate windows

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
            "dataset_size",
        ]
    )

for ds_num, ds_name in enumerate(all_datasets):
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    dataset_term = dataset_properties_map[ds_name]["term"]
    for term in ["short", "medium", "long"]:
        if (term == "medium" or term == "long") and dataset_term == "short":
            continue

        ds_freq = dataset_properties_map[ds_name]["frequency"]
        ds_config = f"{ds_name}/{ds_freq}/{term}"

        # Keep multivariate data multivariate so Toto 2.0 uses its variate-attention path
        dataset = Dataset(name=ds_name, term=term, to_univariate=False, storage_env_var="BOOM")
        target_dim = dataset.target_dim
        season_length = get_seasonality(dataset.freq)
        dataset_size = len(dataset.test_data)
        print(f"  term={term} target_dim={target_dim} pred_len={dataset.prediction_length} size={dataset_size}")

        gts_config = Toto2GluonTSModelConfig(
            prediction_length=dataset.prediction_length,
            context_length=CONTEXT_LENGTH,
            target_dim=target_dim,
        )
        gts_model = Toto2GluonTSModel(toto_model, gts_config).to(DEVICE).eval()
        predictor = gts_model.create_predictor(batch_size=BATCH_SIZE, device=DEVICE)

        res = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=True,
            seasonality=season_length,
        )

        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    model_name,
                    res["MSE[mean]"].iloc[0],
                    res["MSE[0.5]"].iloc[0],
                    res["MAE[0.5]"].iloc[0],
                    res["MASE[0.5]"].iloc[0],
                    res["MAPE[0.5]"].iloc[0],
                    res["sMAPE[0.5]"].iloc[0],
                    res["MSIS"].iloc[0],
                    res["RMSE[mean]"].iloc[0],
                    res["NRMSE[mean]"].iloc[0],
                    res["ND[0.5]"].iloc[0],
                    res["mean_weighted_sum_quantile_loss"].iloc[0],
                    dataset_properties_map[ds_name]["domain"],
                    dataset_properties_map[ds_name]["num_variates"],
                    dataset_size,
                ]
            )
        print(f"  wrote row for {ds_config}")

## Results

In [ ]:
import pandas as pd
df = pd.read_csv(output_dir + "/all_results.csv")
df